In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/annotated/checkpoints/pre_cell_2.pickle

trying: ['load_lineitem']
me:  1
trying: ['load_part']
me:  2
trying: ['pd']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['lineitem']


me:  1
trying: ['part']
me:  2
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_part
Declaring variable part


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Define container and ship mode lists
SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
SHIPMODES = ["AIR", "AIRREG"]

# 1) Collapse the three disjoint between() calls into one: 4–36
# 2) Use __getitem__ (df["col"]) instead of attribute access to cut down NDFrame.__getattr__ overhead
fline = lineitem[
    lineitem["L_QUANTITY"].between(4, 36)
    & (lineitem["L_SHIPINSTRUCT"] == "DELIVER IN PERSON")
    & lineitem["L_SHIPMODE"].isin(SHIPMODES)
]

# Filter part in one shot
fpart = part[
    ((part["P_BRAND"] == "Brand#31") & part["P_CONTAINER"].isin(SM_SMALL) & part["P_SIZE"].between(1, 5))
    | ((part["P_BRAND"] == "Brand#43") & part["P_CONTAINER"].isin(MED_CONTAINER) & part["P_SIZE"].between(1, 10))
    | ((part["P_BRAND"] == "Brand#43") & part["P_CONTAINER"].isin(LG_CONTAINER) & part["P_SIZE"].between(1, 15))
]

# Join
df = fline.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

# Final multi‐range filter in one mask
mask = (
    (df["P_BRAND"] == "Brand#31") & df["P_CONTAINER"].isin(SM_SMALL) & df["L_QUANTITY"].between(4, 14) & (df["P_SIZE"] <= 5)
    | (df["P_BRAND"] == "Brand#43") & df["P_CONTAINER"].isin(MED_CONTAINER) & df["L_QUANTITY"].between(15, 25) & (df["P_SIZE"] <= 10)
    | (df["P_BRAND"] == "Brand#43") & df["P_CONTAINER"].isin(LG_CONTAINER) & df["L_QUANTITY"].between(26, 36) & (df["P_SIZE"] <= 15)
)

df = df[mask]

# Compute total on GPU
total = (df["L_EXTENDEDPRICE"] * (1.0 - df["L_DISCOUNT"])) .sum()

CPU times: user 111 ms, sys: 15.5 ms, total: 127 ms
Wall time: 133 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q19/rewritten/o4_mini_high/checkpoints/post_cell_2_try_2.pickle

migration speed (bps): 781466140.8177763
---------------------------
variables to migrate:
load_lineitem 144
fline 48
STORAGE_OPTS 64
lineitem 48
pd 72
mask 48
total 32
part 48
SHIPMODES 72
LG_CONTAINER 88
fpart 48
DATA_ROOT 80
SM_SMALL 88
MED_CONTAINER 88
tpch_parent 54
load_part 144
df 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q19/rewritten/o4_mini_high/checkpoints/post_cell_2_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['SM_SMALL', 'total', 'LG_CONTAINER', 'fline', 'SHIPMODES', 'MED_CONTAINER', 'df', 'mask', 'fpart']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q19/opt_cell_exec_info_2_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/annotated/checkpoints/post_cell_2.pickle
assert total_opt == total

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['LGBOX']
me:  3
trying: ['SMCASE']
me:  3
trying: ['Brand31']
me:  3
trying: ['lsel']
me:  3
trying: ['total']
me:  3
trying: ['LGCASE']
me:  3
trying: ['LGPKG']
me:  3
trying: ['SMPACK']
me:  3
trying: ['jn']
me:  3
trying: ['load_part']
me:  2
trying: ['pd']
me:  0
trying: ['part']
me:  2
trying: ['fpart']
me:  3
trying: ['jnsel']
me:  3
trying: ['flineitem']
me:  3
trying: ['MEDPKG']
me:  3
trying: ['load_lineitem']
me:  1
trying: ['Brand43']
me:  3
trying: ['MEDBOX']
me:  3
trying: ['lineitem']


me:  1
trying: ['LGPACK']
me:  3
trying: ['DELIVERINPERSON']
me:  3
trying: ['SMPKG']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['MEDPACK']
me:  3
trying: ['psel']
me:  3
trying: ['SMBOX']
me:  3
trying: ['AIR']
me:  3
trying: ['MEDBAG']
me:  3
trying: ['DATA_ROOT']
me:  0
trying: ['AIRREG']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['orig_output']
me:  4


Declaring variable pd
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_part
Declaring variable part
Declaring variable LGBOX
Declaring variable SMCASE
Declaring variable Brand31
Declaring variable lsel
Declaring variable total
Declaring variable LGCASE
Declaring variable LGPKG
Declaring variable SMPACK
Declaring variable jn
Declaring variable fpart
Declaring variable jnsel
Declaring variable flineitem
Declaring variable MEDPKG
Declaring variable Brand43
Declaring variable MEDBOX
Declaring variable LGPACK
Declaring variable DELIVERINPERSON
Declaring variable SMPKG
Declaring variable MEDPACK
Declaring variable psel
Declaring variable SMBOX
Declaring variable AIR
Declaring variable MEDBAG
Declaring variable AIRREG
Declaring variable orig_output
